# 과제 1 — SQL 기초 (20점)

**데이터베이스 관리 · 과제 1**

- **제출물**: 이 노트북 파일 하나 (모든 셀을 실행한 상태로 저장)
- **마감**: 9주차 수업 후 1주일, LMS 제출
- **배점**: 자동 확인 12점 + 결과 해석 5점 + 실행 완료 · 형식 3점 = 20점
- **범위**: 7주차(SELECT · WHERE · ORDER BY) + 8주차(집계 · GROUP BY · HAVING · JOIN)

### 하는 방법
1. 아래 **준비 셀**에 본인 학번과 이름을 적고 실행합니다. 학번에 따라 **분석할 질환이 배정**됩니다.
2. 각 문항의 `%%sql` 셀에 SQL을 완성하고 실행합니다. `___` 부분을 채우면 됩니다.
3. 바로 다음 셀의 `check("Q1", _)` 를 실행하면 정답 여부가 표시됩니다.
4. 마지막 **해석 문항 2개**는 직접 문장으로 작성합니다.

> 결과 열의 **이름과 순서**도 채점에 들어갑니다. 문항에 적힌 대로 `AS` 이름을 붙이세요.

## 0. 준비

In [ ]:
# ── 0. 준비: 이 셀을 먼저 실행하세요 (Shift+Enter) ───────────────────────────
import os, sys, urllib.request, hashlib
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand; break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "jupysql", "duckdb-engine"])

import duckdb, pandas as pd
pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
conn = duckdb.connect(DB, read_only=True)
%sql conn --alias aact

# ── 학번과 이름을 입력하세요 ────────────────────────────────────────────────
학번 = "20260000"     # ← 본인 학번으로 바꾸세요
이름 = "홍길동"        # ← 본인 이름으로 바꾸세요

MESH = ["Diabetes Mellitus, Type 2", "Hypertension", "Asthma", "Breast Neoplasms", "Depressive Disorder",
        "Stroke", "Heart Failure", "Pulmonary Disease, Chronic Obstructive", "Renal Insufficiency, Chronic",
        "Alzheimer Disease", "Obesity", "COVID-19", "Osteoarthritis", "Sepsis", "Pneumonia"]
MY_MESH = MESH[int(hashlib.md5(str(학번).encode()).hexdigest(), 16) % len(MESH)]
print(f"{이름}({학번}) 님의 배정 질환:  {MY_MESH}")
print("→ 아래 문항에서 '배정 질환' 이라고 나오면 이 값을 쓰세요.")

def _sig(df):
    d = df.copy()
    d.columns = [str(c).lower() for c in d.columns]
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(4)
    d = d.astype(str).sort_values(by=list(d.columns)).reset_index(drop=True)
    return hashlib.md5(d.to_csv(index=False).encode()).hexdigest()[:12]

def check(name, df):
    """직전 %%sql 결과를 _ 로 넘겨 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print("알 수 없는 문항:", name); return
    a = ANSWERS[name]
    exp = a["by_mesh"].get(MY_MESH) if a.get("by_mesh") else a
    if not isinstance(df, pd.DataFrame):
        print(f"❌ {name}  결과가 표가 아닙니다. %%sql 셀을 먼저 실행한 뒤 바로 다음 셀에서 check 하세요."); return
    if _sig(df) == exp["sig"]:
        print(f"✅ {name} 정답!  (행 {len(df)}개)")
    else:
        msg = f"❌ {name} 아직 다릅니다.  기대 행 수 {exp['rows']}개, 현재 {len(df)}개."
        if a.get("hint"): msg += "  힌트: " + a["hint"]
        print(msg)

def 제출확인():
    done = [k for k in ANSWERS if k.startswith("Q")]
    print(f"문항 {len(done)}개.  모든 셀을 실행한 뒤 파일을 저장해 제출하세요.")
    print(f"제출자: {이름} ({학번}) / 배정 질환: {MY_MESH}")

In [ ]:
ANSWERS = {
 "Q1": {
  "sig": "be0159f8b22a",
  "rows": 5,
  "hint": "ORDER BY start_date DESC, nct_id 로 정렬하고 LIMIT 5"
 },
 "Q2": {
  "sig": "2c1777bcb143",
  "rows": 1,
  "hint": "overall_status = 'RECRUITING'"
 },
 "Q3": {
  "sig": "dde5fc01a6a9",
  "rows": 2,
  "hint": "DISTINCT 와 ORDER BY"
 },
 "Q4": {
  "sig": "23e0c4a83794",
  "rows": 1,
  "hint": "AND 와 IN 을 함께"
 },
 "Q5": {
  "sig": "c58102c6a9ad",
  "rows": 1,
  "hint": "ILIKE 로 대소문자 무시"
 },
 "Q6": {
  "sig": "aa3358328438",
  "rows": 1,
  "hint": "IS NULL 사용"
 },
 "Q7": {
  "sig": "120774014f9f",
  "rows": 9,
  "hint": "GROUP BY phase, ORDER BY n DESC"
 },
 "Q8": {
  "sig": "a0eb9485d959",
  "rows": 2,
  "hint": "AVG 는 ROUND 로 감싸고, MEDIAN 도 함께"
 },
 "Q9": {
  "sig": "d1b993762472",
  "rows": 9,
  "hint": "WHERE removed = false, HAVING COUNT(DISTINCT nct_id) >= 1500"
 },
 "Q10": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "2a097adc8672",
    "rows": 1
   },
   "Hypertension": {
    "sig": "3a83d56e33b3",
    "rows": 1
   },
   "Asthma": {
    "sig": "42dc7cb7c848",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "de4e3075c3a1",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "eeaa5a5d8b54",
    "rows": 1
   },
   "Stroke": {
    "sig": "46d59ee70766",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "b04daacc2ae4",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "6b96a578c2d8",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "eeaa5a5d8b54",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "c756da343326",
    "rows": 1
   },
   "Obesity": {
    "sig": "36187ee2daad",
    "rows": 1
   },
   "COVID-19": {
    "sig": "05ad9322bb6a",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "fe7a133358cd",
    "rows": 1
   },
   "Sepsis": {
    "sig": "70973c9edb30",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "bd8fba07b3f8",
    "rows": 1
   }
  },
  "hint": "browse_conditions 에서 COUNT(DISTINCT nct_id)"
 },
 "Q11": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "2edb4b700bc5",
    "rows": 9
   },
   "Hypertension": {
    "sig": "3b5566ab1349",
    "rows": 9
   },
   "Asthma": {
    "sig": "2012c253eab7",
    "rows": 9
   },
   "Breast Neoplasms": {
    "sig": "5101a4553e9d",
    "rows": 9
   },
   "Depressive Disorder": {
    "sig": "c8beed941eb5",
    "rows": 9
   },
   "Stroke": {
    "sig": "97e6fb5ac246",
    "rows": 9
   },
   "Heart Failure": {
    "sig": "d0b44e016e92",
    "rows": 9
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "77bfb711732c",
    "rows": 9
   },
   "Renal Insufficiency, Chronic": {
    "sig": "69c3a43c7930",
    "rows": 9
   },
   "Alzheimer Disease": {
    "sig": "acb635d7cc43",
    "rows": 9
   },
   "Obesity": {
    "sig": "5734840ea216",
    "rows": 9
   },
   "COVID-19": {
    "sig": "46f6dc4b3d3c",
    "rows": 9
   },
   "Osteoarthritis": {
    "sig": "436c8d6c9b32",
    "rows": 9
   },
   "Sepsis": {
    "sig": "2f2279835e21",
    "rows": 9
   },
   "Pneumonia": {
    "sig": "da6ef813c648",
    "rows": 9
   }
  },
  "hint": "JOIN 후 GROUP BY overall_status, COUNT(DISTINCT s.nct_id)"
 },
 "Q12": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "0d18af733fd7",
    "rows": 7
   },
   "Hypertension": {
    "sig": "5c2b4ee95f69",
    "rows": 6
   },
   "Asthma": {
    "sig": "8316aee1ece7",
    "rows": 6
   },
   "Breast Neoplasms": {
    "sig": "230fc55f9921",
    "rows": 6
   },
   "Depressive Disorder": {
    "sig": "67cbb570f87b",
    "rows": 7
   },
   "Stroke": {
    "sig": "bc3dae7816c1",
    "rows": 7
   },
   "Heart Failure": {
    "sig": "75c503a5abf1",
    "rows": 7
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "ab864c785916",
    "rows": 5
   },
   "Renal Insufficiency, Chronic": {
    "sig": "240ea20d6345",
    "rows": 4
   },
   "Alzheimer Disease": {
    "sig": "9a86843901a2",
    "rows": 6
   },
   "Obesity": {
    "sig": "3ef194afcafd",
    "rows": 7
   },
   "COVID-19": {
    "sig": "fe492993c782",
    "rows": 7
   },
   "Osteoarthritis": {
    "sig": "1f6dc4f9392b",
    "rows": 6
   },
   "Sepsis": {
    "sig": "30132d7df6db",
    "rows": 5
   },
   "Pneumonia": {
    "sig": "762b6a819370",
    "rows": 7
   }
  },
  "hint": "표 세 개를 연결하고 lead 만, COUNT(DISTINCT s.nct_id)"
 }
}

## 1부. 조회 기초 (7주차 범위)  — 6문항

**Q1.** 가장 최근에 시작한 시험 5건의 `nct_id`, `brief_title`, `start_date` 를 시작일 내림차순으로 조회하세요. (같은 날짜면 `nct_id` 오름차순)

In [ ]:
%%sql
-- Q1
SELECT nct_id, brief_title, start_date
FROM studies
ORDER BY ___
LIMIT ___

In [ ]:
check("Q1", _)

**Q2.** 모집 중(`RECRUITING`)인 시험 수를 `n` 으로 세세요.

In [ ]:
%%sql
-- Q2
SELECT COUNT(*) AS n
FROM studies
WHERE ___

In [ ]:
check("Q2", _)

**Q3.** `study_type` 에 어떤 값이 있는지 중복 없이 알파벳 순으로 조회하세요.

In [ ]:
%%sql
-- Q3
SELECT ___ study_type
FROM studies
___ study_type

In [ ]:
check("Q3", _)

**Q4.** 등록 인원이 500명 이상이면서 임상 단계가 PHASE3 또는 PHASE4 인 시험 수를 `n` 으로 세세요. (IN 사용)

In [ ]:
%%sql
-- Q4
SELECT COUNT(*) AS n
FROM studies
WHERE ___ AND phase IN (___)

In [ ]:
check("Q4", _)

**Q5.** 제목에 nurse 가 들어간 시험 수를 `n` 으로 세세요. **대소문자를 구분하지 않아야 합니다.**

In [ ]:
%%sql
-- Q5
SELECT COUNT(*) AS n
FROM studies
WHERE brief_title ___ '%nurse%'

In [ ]:
check("Q5", _)

**Q6.** 완료(`COMPLETED`)되었지만 결과 게시일(`results_first_posted_date`)이 없는 시험 수를 `n` 으로 세세요.

In [ ]:
%%sql
-- Q6
SELECT COUNT(*) AS n
FROM studies
WHERE overall_status = 'COMPLETED'
  AND ___

In [ ]:
check("Q6", _)

## 2부. 집계와 GROUP BY (8주차 범위)  — 3문항

**Q7.** 임상 단계(`phase`)별 시험 수를 `n` 으로 세고 많은 순으로 정렬하세요.

In [ ]:
%%sql
-- Q7
SELECT phase, COUNT(*) AS n
FROM studies
GROUP BY ___
ORDER BY ___

In [ ]:
check("Q7", _)

**Q8.** 연구 유형(`study_type`)별 등록 인원의 평균(`평균`, 반올림)과 중앙값(`중앙값`)을 구하고 `study_type` 순으로 정렬하세요.

In [ ]:
%%sql
-- Q8
SELECT study_type, ___ AS 평균, ___ AS 중앙값
FROM studies
GROUP BY study_type
ORDER BY study_type

In [ ]:
check("Q8", _)

**Q9.** 국가(`국가`)별 시험 수(`시험수`)를 구하되 제외된 국가 행(`removed`)은 빼고, 1500건 이상인 국가만 많은 순으로 조회하세요.

In [ ]:
%%sql
-- Q9
SELECT name AS 국가, COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE ___
GROUP BY name
HAVING ___
ORDER BY 시험수 DESC

In [ ]:
check("Q9", _)

## 3부. 배정 질환 분석 (JOIN)  — 3문항

아래 세 문항은 **본인에게 배정된 질환**(준비 셀에 표시된 `MY_MESH`)을 대상으로 합니다.
SQL 안에 질환 이름을 **직접 따옴표로 적으세요.** (예: `WHERE mesh_term = 'Hypertension'`)

**Q10.** 배정 질환을 대상으로 한 임상시험이 몇 건인지 `n` 으로 세세요. (`browse_conditions`, 중복 제거)

In [ ]:
%%sql
-- Q10
SELECT COUNT(DISTINCT nct_id) AS n
FROM browse_conditions
WHERE mesh_term = ___

In [ ]:
check("Q10", _)

**Q11.** 배정 질환 시험을 진행 상태(`상태`)별로 나누어 시험 수(`시험수`)를 많은 순으로 조회하세요. (studies 와 JOIN)

In [ ]:
%%sql
-- Q11
SELECT s.overall_status AS 상태, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON ___
WHERE b.mesh_term = ___
GROUP BY s.overall_status
ORDER BY 시험수 DESC

In [ ]:
check("Q11", _)

**Q12.** 배정 질환 시험의 **주관 스폰서**(`lead`) 유형(`스폰서유형`)별 시험 수(`시험수`)를 많은 순으로 조회하세요. (studies · browse_conditions · sponsors)

In [ ]:
%%sql
-- Q12
SELECT sp.agency_class AS 스폰서유형, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN sponsors AS sp ON ___
WHERE b.mesh_term = ___
  AND sp.lead_or_collaborator = ___
GROUP BY sp.agency_class
ORDER BY 시험수 DESC

In [ ]:
check("Q12", _)

## 4부. 결과 해석 (5점)

아래 두 문항은 자동 채점하지 않습니다. **직접 문장으로** 작성하세요. 각 3~5문장이면 충분합니다.

---

**해석 1 (3점).** Q11 결과를 보고, 본인에게 배정된 질환의 임상시험 현황을 설명하세요.
가장 많은 상태가 무엇이고 그것이 무엇을 뜻하는지, 그리고 이 숫자를 해석할 때 주의할 점을 함께 적으세요.

*(아래 칸을 두 번 눌러 편집한 뒤 작성하세요)*

> 답:

---

**해석 2 (2점).** Q9에서 `WHERE removed = false` 조건을 빼면 결과가 어떻게 달라질까요?
왜 그런지, 그리고 이런 조건을 빠뜨리면 실제 분석에서 어떤 문제가 생길 수 있는지 적으세요.

> 답:

---

## 5. 제출 전 확인

In [ ]:
제출확인()

제출 전 체크리스트

- [ ] 준비 셀에 **학번과 이름**을 입력했다
- [ ] 12개 문항이 모두 ✅ 로 표시된다
- [ ] 해석 1 · 2를 문장으로 작성했다
- [ ] **모든 셀을 위에서부터 실행**한 상태로 저장했다
- [ ] 파일 이름을 `과제1_학번_이름.ipynb` 로 바꾸었다